# Módulo 06 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Esta lista percorre o módulo inteiro, do primeiro `GET` até a API autenticada.

## Como usar

| | |
|---|---|
| 🟢 **Aquecimento** | 1–10 · uma ideia por exercício |
| 🟡 **Construção** | 11–26 · combinam dois ou três conceitos |
| 🔴 **Integração** | 27–38 · perto de código de produção |
| 🏗️ **Projeto** | A Atlas API v1 |

**Regras de casa:**

1. Escreva antes de rodar. Preveja a resposta e o status code.
2. Quando errar, **entenda antes de corrigir**. O erro é a aula.
3. Não pule os 🔴 — são eles que separam "sei FastAPI" de "sei fazer API".

> 💡 As respostas não estão aqui de propósito. Cada exercício é verificável: rode e compare com o que você previu.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação — Lista de Exercícios do Módulo 06
# ═══════════════════════════════════════════════════════════════
import json
import os
import secrets
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("sqlalchemy", "sqlalchemy"),
               ("pydantic-settings", "pydantic_settings"),
               ("pyjwt", "jwt"), ("python-multipart", "multipart"),
               ("bcrypt", "bcrypt")]:
    _garantir(_p, _m)

import fastapi
from fastapi.testclient import TestClient

print(f"✅ FastAPI {fastapi.__version__}")

os.environ.setdefault("ATLAS_CHAVE_SECRETA", secrets.token_urlsafe(48))
os.environ.setdefault("ATLAS_AMBIENTE", "desenvolvimento")

BASE = Path("lista_06").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
BASE.mkdir(parents=True)
sys.path = [str(BASE)] + [p for p in sys.path if p != str(BASE)]
print(f"📁 {BASE}")


def req(cliente, metodo: str, caminho: str, mostrar_corpo=True, **kwargs):
    resposta = getattr(cliente, metodo.lower())(caminho, **kwargs)
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    print(f"{cor} {metodo.upper():<7} {caminho:<40} → {resposta.status_code}")
    if kwargs.get("json") is not None:
        corpo = json.dumps(kwargs["json"], ensure_ascii=False)
        print(f"   envio  : {corpo[:120]}{'...' if len(corpo) > 120 else ''}")
    if mostrar_corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            linhas = texto.splitlines()
            for linha in linhas[:12]:
                print(f"   {linha}")
            if len(linhas) > 12:
                print(f"   ... (+{len(linhas) - 12} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:200]}")
    print()
    return resposta


# ═══════════════════════════════════════════════════════════════
#  Dados de apoio — o catálogo da Aurora
# ═══════════════════════════════════════════════════════════════
CATALOGO = [
    {"sku": "NB-DELL-15", "nome": "Notebook Dell Inspiron 15", "categoria": "Notebooks",
     "preco": 2599.90, "custo": 2120.00, "estoque": 14},
    {"sku": "NB-LEN-IP3", "nome": "Notebook Lenovo IdeaPad 3", "categoria": "Notebooks",
     "preco": 2199.00, "custo": 1790.00, "estoque": 6},
    {"sku": "MO-LG-24UW", "nome": "Monitor LG 24 UltraWide", "categoria": "Monitores",
     "preco": 1199.00, "custo": 920.00, "estoque": 31},
    {"sku": "MO-SAM-27C", "nome": "Monitor Samsung 27 Curvo", "categoria": "Monitores",
     "preco": 1549.00, "custo": 1210.00, "estoque": 0},
    {"sku": "PE-LOG-M170", "nome": "Mouse Logitech M170", "categoria": "Periféricos",
     "preco": 89.90, "custo": 52.00, "estoque": 120},
    {"sku": "PE-RED-K552", "nome": "Teclado Redragon K552", "categoria": "Periféricos",
     "preco": 249.90, "custo": 168.00, "estoque": 8},
    {"sku": "AR-KING-1TB", "nome": "SSD Kingston NV2 1TB", "categoria": "Armazenamento",
     "preco": 429.00, "custo": 305.00, "estoque": 47},
    {"sku": "AR-WD-2TB", "nome": "HD Western Digital 2TB", "categoria": "Armazenamento",
     "preco": 389.00, "custo": 288.00, "estoque": 23},
]

print(f"✅ `req()` pronta · {len(CATALOGO)} produtos em CATALOGO")

---

# 🟢 Aquecimento (1–10)

Uma ideia por exercício. Se algum travar você por mais de 10 minutos, releia a aula correspondente.

### 1 · Primeiro servidor

Crie um `FastAPI` com `GET /saude` devolvendo `{"status": "ok"}` e `GET /versao` devolvendo `{"versao": "1.0.0"}`. Teste as duas com `TestClient` e confirme o status `200`.

In [ ]:
# 1

### 2 · Path parameter tipado

Crie `GET /pedidos/{pedido_id}` com `pedido_id: int`.

Chame com `/pedidos/42` e com `/pedidos/abc`. Antes de rodar, escreva num comentário: qual status cada uma devolve, e **quem** gerou o erro.

In [ ]:
# 2

### 3 · A ordem das rotas importa

Declare, **nesta ordem**, `GET /produtos/{sku}` e depois `GET /produtos/destaques`.

Chame `/produtos/destaques` e explique o resultado. Depois inverta a ordem e chame de novo.

In [ ]:
# 3

### 4 · Query params com validação

`GET /produtos` com `limite` (1 a 100, padrão 20) e `pagina` (≥ 1, padrão 1), usando `Query()`.

Teste `limite=500` e `pagina=0`.

In [ ]:
# 4

### 5 · Status code correto

Crie quatro rotas que devolvam, cada uma, um destes: `201`, `204`, `404`, `409`. Escreva num comentário em que situação real cada um se aplica.

In [ ]:
# 5

### 6 · Corpo com Pydantic

Modele `ProdutoCriar` com `sku` (mínimo 5 caracteres), `nome` (mínimo 3), `preco` (> 0) e `estoque` (≥ 0, padrão 0). Crie `POST /produtos` e teste com um corpo válido e um com três erros simultâneos.

In [ ]:
# 6

### 7 · Conversão automática

Envie `{"preco": "1199.00", "estoque": "31"}` para a rota do exercício 6 e imprima o **tipo** de cada campo depois da validação. Depois envie `{"preco": "caro"}`.

In [ ]:
# 7

### 8 · 🔒 `response_model`

Crie um modelo `ProdutoBanco` com `custo` e `fornecedor`, e um `ProdutoPublico` sem eles.

Faça duas rotas devolvendo o **mesmo objeto**: uma sem `response_model` e outra com. Compare as respostas.

In [ ]:
# 8

### 9 · `HTTPException`

Crie `GET /produtos/{sku}` que devolva `404` com uma mensagem útil quando o SKU não existir no `CATALOGO`. Teste com um SKU válido e um inválido.

In [ ]:
# 9

### 10 · Documentação automática

Monte uma app com três rotas em duas *tags* diferentes. Leia `/openapi.json` e imprima uma tabela: método, caminho, tag, códigos de resposta documentados.

In [ ]:
# 10

---

# 🟡 Construção (11–26)

Agora os conceitos se combinam.

### 11 · Validadores que normalizam

Modele `ProdutoCriar` com validadores que:

- coloquem o `sku` em maiúsculas e removam espaços
- colapsem espaços múltiplos do `nome`
- ordenem e removam duplicatas das `tags`

⚠️ Um deles precisa de `mode="before"`. Descubra qual e explique por quê num comentário.

In [ ]:
# 11

### 12 · `model_validator`

Adicione ao modelo do exercício 11 a regra `preco >= custo`.

Depois responda num comentário: por que ela **não** pode ser um `field_validator`?

In [ ]:
# 12

### 13 · Corpo aninhado

Modele `PedidoCriar` com `entrega: Endereco` e `itens: list[ItemPedido]`.

Envie um corpo com erro no **segundo item** e no **CEP**, e imprima o `loc` de cada erro.

In [ ]:
# 13

### 14 · A família de modelos

Para `Cliente`, escreva `ClienteBase`, `ClienteCriar`, `ClienteAtualizar` e `ClienteResposta`.

Num comentário, justifique cada uma das quatro.

In [ ]:
# 14

### 15 · `PATCH` correto

Implemente `PATCH /produtos/{sku}` com `model_dump(exclude_unset=True)`.

Depois faça uma versão **sem** o `exclude_unset` e mostre o bug: alterar só o preço apaga o nome.

In [ ]:
# 15

### 16 · Exceções de domínio

Crie `AtlasError` e três filhas. Escreva `@app.exception_handler` para cada uma.

**Requisito:** nenhuma função de rota pode importar `HTTPException`.

In [ ]:
# 16

### 17 · Padronizar o 422

Escreva um handler para `RequestValidationError` que devolva:

```json
{"codigo": "validacao_falhou", "campos": [{"campo": "preco", "erro": "..."}]}
```

In [ ]:
# 17

### 18 · 🔴 O traceback não vaza

Crie uma rota que divida por zero e um handler global para `Exception`.

Prove que a resposta **não** contém traceback, caminho de arquivo nem nome de biblioteca, e que o detalhe foi para o log.

In [ ]:
# 18

### 19 · Dependência reutilizável

Crie `paginacao(pagina, por_pagina)` como dependência e use em três rotas. Confirme no `openapi.json` que os parâmetros aparecem documentados nas três.

In [ ]:
# 19

### 20 · Dependências aninhadas

Monte a corrente `obter_token` → `usuario_atual` → `exigir_admin`.

Teste os quatro casos: sem token, token inválido, usuário sem permissão, usuário admin. Confirme que os status são `401`, `401`, `403`, `200`.

In [ ]:
# 20

### 21 · Cache de dependência

Prove que uma dependência usada três vezes na mesma rota executa **uma** vez. Depois prove que executa três vezes com `use_cache=False`. E prove que o cache não atravessa requisições.

In [ ]:
# 21

### 22 · `APIRouter`

Quebre uma app de 6 rotas em dois routers com `prefix` e `tags`. Confirme que os caminhos finais estão corretos no `openapi.json`.

In [ ]:
# 22

### 23 · 🔴 Sessão por requisição

Escreva `get_sessao` com `yield` e `try/finally`.

Depois escreva uma versão **sem** o `close()`, configure `pool_size=2, max_overflow=0` e faça 20 requisições. Mostre o erro e explique a mensagem.

In [ ]:
# 23

### 24 · SQLAlchemy + Pydantic

Crie um modelo `Produto` (SQLAlchemy) e um `ProdutoResposta` (Pydantic) com `from_attributes=True`.

Retorne o objeto do banco direto da rota e mostre que o FastAPI o converte — e filtra o `custo`.

In [ ]:
# 24

### 25 · `lifespan`

Use `lifespan` para criar as tabelas na subida e imprimir a contagem de produtos na descida.

Prove com `with TestClient(app) as c:` que as duas partes executam.

In [ ]:
# 25

### 26 · `dependency_overrides`

Substitua `get_sessao` por um banco SQLite em memória.

⚠️ Você vai precisar de `poolclass=StaticPool`. Descubra o erro sem ele antes de aplicar a correção.

In [ ]:
# 26

---

# 🔴 Integração (27–38)

Estes exercícios são o que separa um tutorial de um sistema.

### 27 · 🔴 Transação atômica

Implemente `POST /pedidos` que baixe estoque de vários produtos.

**Prova obrigatória:** um pedido com 3 itens onde o **terceiro** não tem estoque não pode alterar o estoque dos dois primeiros. Imprima o estoque antes e depois.

In [ ]:
# 27

### 28 · Camadas de verdade

Separe `repositorio.py`, `servicos.py` e `rotas/`.

**Regras:**

- o repositório nunca chama `commit`
- o serviço nunca importa `fastapi`
- nenhuma rota passa de 5 linhas

Escreva um teste que verifique a segunda regra lendo o arquivo.

In [ ]:
# 28

### 29 · N+1

Liste 20 pedidos com itens e produtos. Meça o número de consultas com `echo=True`, com e sem `selectinload`.

Reporte os dois números e explique a diferença.

In [ ]:
# 29

### 30 · 🔴 Configuração validada

Escreva uma `Config` com `BaseSettings` que **recuse** subir se a chave secreta tiver menos de 32 caracteres ou for um valor de exemplo.

Mostre as três mensagens de erro.

In [ ]:
# 30

### 31 · 🔴 Hash de senha

Implemente cadastro e login com bcrypt.

Prove três coisas: hashes diferentes para a mesma senha, verificação funcionando em ambos, e que a senha em texto puro não aparece em nenhuma resposta nem em nenhum log.

In [ ]:
# 31

### 32 · Anatomia do JWT

Emita um token e **leia a carga sem a chave**.

Depois altere o campo `papel` na carga, remonte o token e prove que a verificação o recusa.

In [ ]:
# 32

### 33 · Login completo

Implemente `POST /auth/token` com `OAuth2PasswordRequestForm`.

**Requisito de segurança:** a resposta para "e-mail não existe" e "senha errada" deve ser **byte a byte idêntica**. Prove comparando os dois objetos de resposta.

In [ ]:
# 33

### 34 · 401 vs 403

Monte três papéis e uma rota que exija `operador`.

Produza uma tabela: anônimo, leitor, operador, admin × status recebido. Confirme `401, 403, 200, 200`.

In [ ]:
# 34

### 35 · Middleware de rastreio

Escreva um middleware que adicione `X-Request-ID` e `X-Tempo-ms`.

**Requisito:** se o cliente já enviou um `X-Request-ID`, preserve o valor dele.

In [ ]:
# 35

### 36 · CORS

Configure o CORS para `http://localhost:3000` e prove três coisas:

1. a origem permitida recebe `Access-Control-Allow-Origin`
2. a origem não permitida **não** recebe — mas recebe a resposta
3. o preflight `OPTIONS` responde com os métodos permitidos

Depois explique num comentário por que o item 2 significa que CORS não protege a API.

In [ ]:
# 36

### 37 · 🔴 Lista branca

Implemente ordenação por campo com lista branca.

Mostre três ataques recusados: campo inexistente, campo interno (`custo`) e uma tentativa de injeção de SQL. Explique por que o **segundo** é um vazamento mesmo sem injeção.

In [ ]:
# 37

### 38 · 🔴 Auditoria automática

Escreva uma função que leia o `openapi.json` e falhe se:

- alguma rota de escrita não exigir autenticação
- algum esquema **de resposta** contiver um campo de uma lista de sensíveis

Rode contra a sua API e conserte o que ela apontar.

In [ ]:
# 38

---

# 🏗️ Projeto — Atlas API v1

Chegou a hora de a Aurora ter uma API de verdade.

## O contexto

O Atlas já lê CSV (M01), está versionado (M02), guarda em SQL (M03), tem camadas e tipos (M04) e fala PostgreSQL e MongoDB (M05).

Agora o time do app precisa consumir esses dados. Sua missão: **expor o Atlas como uma API HTTP autenticada, documentada e testável**.

## As dores da Aurora, nas palavras de quem sofre

| Quem | O que diz |
|------|-----------|
| Time do app | *"Preciso listar produtos com filtro e paginação, sem baixar o catálogo inteiro."* |
| Compras | *"Quero registrar entrada de estoque sem abrir o banco."* |
| Diretoria | *"Quero ver margem — mas só eu."* |
| Você | *"Quero dormir sabendo que o estagiário não apaga a base."* |

## O que entregar

```
projeto_Atlas/
├── src/atlas/api/
│   ├── __init__.py
│   ├── aplicacao.py        ← cria o FastAPI, middlewares, handlers
│   ├── config.py           ← BaseSettings
│   ├── dependencias.py     ← sessão, paginação, ordenação, autorização
│   ├── seguranca.py        ← hash e token
│   ├── esquemas.py         ← contratos de entrada e saída
│   └── rotas/
│       ├── autenticacao.py
│       ├── produtos.py
│       ├── pedidos.py
│       └── relatorios.py
├── .env.exemplo
└── docs/API.md
```

## Requisitos obrigatórios

| # | Requisito | Pronto quando |
|---|-----------|---------------|
| 1 | Toda rota tem `response_model` | Nenhum esquema de resposta expõe `custo` |
| 2 | Sessão por requisição | `get_sessao` com `try/finally` |
| 3 | Serviço não conhece HTTP | `grep -r "fastapi" src/atlas/servicos.py` não retorna nada |
| 4 | Erros de domínio traduzidos | Handlers em `aplicacao.py`, não `HTTPException` nas rotas |
| 5 | Segredo fora do código | `.env` ignorado, `.env.exemplo` versionado |
| 6 | Senha com hash | Nenhuma senha em texto no banco |
| 7 | Autorização por papel | leitor / operador / admin |
| 8 | Filtro com lista branca | Ordenação por campo arbitrário recusada |
| 9 | Criação de pedido é atômica | O teste do estoque parcial passa |
| 10 | `docs/API.md` | Exemplos de `curl` para cada rota |

## Rotas mínimas

| Método | Rota | Acesso |
|--------|------|--------|
| `POST` | `/auth/token` | pública |
| `GET` | `/auth/eu` | autenticado |
| `GET` | `/produtos` | autenticado · filtros + paginação |
| `GET` | `/produtos/{sku}` | autenticado |
| `POST` | `/produtos` | operador |
| `PATCH` | `/produtos/{sku}` | operador |
| `DELETE` | `/produtos/{sku}` | admin |
| `POST` | `/pedidos` | operador |
| `GET` | `/pedidos` | autenticado |
| `GET` | `/pedidos/{id}` | autenticado |
| `GET` | `/relatorios/faturamento` | admin |
| `GET` | `/saude` | pública |

> 📋 **O roteiro passo a passo está em `projeto_Atlas/ROTEIRO_M06.md`.** As instruções de ambiente estão no `README.md` do projeto.
>
> 🔴 **Lembre-se:** o esqueleto tem apenas assinaturas e `# TODO`. O código é seu.

## 🧪 Testes de aceitação

Estas células **não** implementam nada — elas **cobram**. Aponte-as para a sua API e faça todas passarem.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Bateria de aceitação — aponte para a SUA aplicação
# ═══════════════════════════════════════════════════════════════
#
#   from atlas.api.aplicacao import criar_app
#   app = criar_app()
#
# Enquanto você não tiver a app pronta, as células abaixo avisam
# e não quebram o notebook.

app = None          # ← troque por: criar_app()

CREDENCIAIS = {
    "admin": ("ana@aurora.com.br", "aurora-admin-2026"),
    "operador": ("bruno@aurora.com.br", "aurora-op-2026"),
    "leitor": ("carla@aurora.com.br", "aurora-leitor-2026"),
}

if app is None:
    print("⏸️  `app` ainda não definida — implemente o projeto e volte aqui.")
else:
    print("▶️  App carregada, pronta para a bateria.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Arcabouço de verificação
# ═══════════════════════════════════════════════════════════════
RESULTADOS = []


def checar(nome: str, condicao: bool, detalhe: str = ""):
    RESULTADOS.append((nome, bool(condicao)))
    print(f"   {'✅' if condicao else '🔴'} {nome}{('  — ' + detalhe) if detalhe else ''}")
    return condicao


def placar():
    passou = sum(1 for _, ok in RESULTADOS if ok)
    total = len(RESULTADOS)
    print(f"\n{'═' * 54}")
    print(f"  {passou}/{total} verificações passaram")
    if total and passou == total:
        print("  🎉 Atlas API v1 aprovada.")
    else:
        for nome, ok in RESULTADOS:
            if not ok:
                print(f"  🔴 pendente: {nome}")
    print("═" * 54)


def autenticar(cliente, papel: str) -> dict:
    email, senha = CREDENCIAIS[papel]
    r = cliente.post("/auth/token", data={"username": email, "password": senha})
    if r.status_code != 200:
        return {}
    return {"Authorization": f"Bearer {r.json()['access_token']}"}


print("✅ `checar()`, `placar()` e `autenticar()` prontos")

In [ ]:
# ── Bateria 1: contrato público ──
RESULTADOS.clear()

if app is None:
    print("⏸️  defina `app` na célula acima")
else:
    cliente = TestClient(app)
    espec = cliente.get("/openapi.json").json()

    checar("GET /saude responde 200",
           cliente.get("/saude").status_code == 200)

    checar("openapi.json tem esquema de segurança",
           bool(espec.get("components", {}).get("securitySchemes")))

    esperadas = {"/auth/token", "/auth/eu", "/produtos", "/produtos/{sku}",
                 "/pedidos", "/pedidos/{id}", "/relatorios/faturamento", "/saude"}
    faltando = esperadas - set(espec["paths"])
    checar("todas as rotas mínimas existem", not faltando, f"faltam {sorted(faltando)}")

    sem_response_model = [
        f"{m.upper()} {c}"
        for c, ms in espec["paths"].items()
        for m, d in ms.items()
        if m in {"get", "post", "patch"}
        and not d["responses"].get("200", d["responses"].get("201", {})).get("content")
    ]
    checar("toda rota de leitura/criação declara response_model",
           not sem_response_model, str(sem_response_model[:3]))

    placar()

In [ ]:
# ── Bateria 2: 🔒 nenhum vazamento de campo interno ──
RESULTADOS.clear()
SENSIVEIS = {"custo", "senha", "senha_hash", "password", "fornecedor"}

if app is None:
    print("⏸️  defina `app` na célula acima")
else:
    # só os esquemas usados em RESPOSTA
    de_saida = set()
    for metodos in espec["paths"].values():
        for detalhe in metodos.values():
            for resposta in detalhe.get("responses", {}).values():
                for tipo in resposta.get("content", {}).values():
                    ref = tipo.get("schema", {}).get("$ref", "")
                    if ref:
                        de_saida.add(ref.rsplit("/", 1)[-1])

    for nome in sorted(de_saida):
        campos = set(espec["components"]["schemas"][nome].get("properties", {}))
        vazou = campos & SENSIVEIS
        # `/relatorios` e rotas de admin podem expor margem — documente a exceção
        checar(f"esquema {nome} sem campo interno", not vazou, str(sorted(vazou)))

    placar()

In [ ]:
# ── Bateria 3: 🔐 autenticação e autorização ──
RESULTADOS.clear()

if app is None:
    print("⏸️  defina `app` na célula acima")
else:
    cliente = TestClient(app)

    checar("sem token → 401",
           cliente.get("/produtos").status_code == 401)
    checar("token inválido → 401",
           cliente.get("/produtos",
                       headers={"Authorization": "Bearer x.y.z"}).status_code == 401)

    r_inexistente = cliente.post("/auth/token",
                                 data={"username": "ninguem@x.com", "password": "a"})
    r_senha_errada = cliente.post("/auth/token",
                                  data={"username": CREDENCIAIS["admin"][0],
                                        "password": "errada"})
    checar("🔴 erro idêntico para usuário e senha errados",
           r_inexistente.status_code == r_senha_errada.status_code
           and r_inexistente.json() == r_senha_errada.json())

    cabecalhos = {p: autenticar(cliente, p) for p in CREDENCIAIS}
    checar("os três papéis conseguem entrar", all(cabecalhos.values()))

    if all(cabecalhos.values()):
        r = cliente.delete("/produtos/NB-DELL-15", headers=cabecalhos["leitor"])
        checar("leitor não apaga produto (403)", r.status_code == 403, f"veio {r.status_code}")

        r = cliente.delete("/produtos/NAO-EXISTE", headers=cabecalhos["admin"])
        checar("admin passa pela autorização (não é 403)", r.status_code != 403,
               f"veio {r.status_code}")

    placar()

In [ ]:
# ── Bateria 4: 🔴 lista branca e validação ──
RESULTADOS.clear()

if app is None:
    print("⏸️  defina `app` na célula acima")
else:
    cab = autenticar(TestClient(app), "leitor")
    cliente = TestClient(app)

    for valor, rotulo in [("custo", "campo interno"),
                          ("inexistente", "campo inexistente"),
                          ("preco; DROP TABLE produtos--", "tentativa de injeção")]:
        r = cliente.get("/produtos", params={"ordenar_por": valor}, headers=cab)
        checar(f"ordenar_por recusa {rotulo}", r.status_code == 422,
               f"veio {r.status_code}")

    r = cliente.get("/produtos", params={"por_pagina": 100000}, headers=cab)
    checar("por_pagina tem teto", r.status_code == 422, f"veio {r.status_code}")

    cab_op = autenticar(cliente, "operador")
    r = cliente.post("/produtos", json={"sku": "ab", "nome": "x", "preco": -1,
                                        "custo": 0}, headers=cab_op)
    checar("produto inválido → 422", r.status_code == 422, f"veio {r.status_code}")

    placar()

In [ ]:
# ── Bateria 5: 🔴 atomicidade do pedido ──
RESULTADOS.clear()

if app is None:
    print("⏸️  defina `app` na célula acima")
else:
    cliente = TestClient(app)
    cab = autenticar(cliente, "operador")

    ALVO_OK = "AR-KING-1TB"       # ajuste para um SKU com estoque folgado
    ALVO_FALHA = "MO-SAM-27C"     # ajuste para um SKU sem estoque

    antes = cliente.get(f"/produtos/{ALVO_OK}", headers=cab)
    if antes.status_code != 200:
        print(f"⏸️  ajuste ALVO_OK — /produtos/{ALVO_OK} devolveu {antes.status_code}")
    else:
        estoque_antes = antes.json()["estoque"]

        r = cliente.post("/pedidos", headers=cab, json={
            "cliente_email": "teste@aurora.com.br", "canal": "site",
            "itens": [{"sku": ALVO_OK, "quantidade": 1},
                      {"sku": ALVO_FALHA, "quantidade": 99999}]})
        checar("pedido inviável é recusado", r.status_code in (409, 422),
               f"veio {r.status_code}")

        depois = cliente.get(f"/produtos/{ALVO_OK}", headers=cab).json()["estoque"]
        checar("🔴 estoque do item viável NÃO foi alterado",
               depois == estoque_antes, f"{estoque_antes} → {depois}")

    placar()

> 🎯 **A bateria 5 é a mais importante do módulo.**
>
> As outras verificam que você seguiu boas práticas. Esta verifica que o seu sistema **não perde mercadoria**. É o tipo de erro que não aparece em nenhum log, não gera exceção e só é descoberto no inventário — quando já é tarde.

---

## 🎓 Autoavaliação

Responda com sinceridade. As lacunas apontam o que reler antes do Módulo 07.

| # | Consigo… | ✅ |
|---|----------|---|
| 1 | Explicar a diferença entre `PUT` e `PATCH` sem consultar | |
| 2 | Escolher entre `400`, `401`, `403`, `404`, `409` e `422` | |
| 3 | Explicar por que a ordem das rotas importa | |
| 4 | Dizer quando `field_validator` precisa de `mode="before"` | |
| 5 | Explicar o que `response_model` **faz**, não só documenta | |
| 6 | Escrever um `PATCH` correto sem consultar | |
| 7 | Explicar por que o serviço não deve importar `fastapi` | |
| 8 | Escrever `get_sessao` de memória, com `try/finally` | |
| 9 | Explicar o que quebra sem o `close()` | |
| 10 | Explicar quem faz `commit` e por quê | |
| 11 | Descrever o problema N+1 e sua correção | |
| 12 | Usar `dependency_overrides` num teste | |
| 13 | Explicar por que `sha256` não serve para senha | |
| 14 | Explicar que JWT é assinado, não criptografado | |
| 15 | Distinguir `401` de `403` com um exemplo cada | |
| 16 | Explicar por que CORS não protege a API | |
| 17 | Justificar a lista branca em dois motivos diferentes | |
| 18 | Provar que um pedido inviável não altera estoque | |

**Menos de 14?** Releia as aulas correspondentes antes de seguir.

---

### ➡️ Próximo módulo

**Módulo 07 — APIs na Prática.** Consumir APIs de terceiros, testar a sua com `pytest`, lidar com falha de rede, tentativas e limites de taxa.

A API que você acabou de construir vira o objeto de teste.